# Step 4 — ERP / P300 Unexpected-Behavior Diagnostic

## Objective
Investigate whichever subjects are currently marked `UNEXPECTED_PATTERN` by the refreshed Step 4 analysis.

This notebook uses the **already-preprocessed clean epochs**. It does not change labels,
polarity, reference, filtering, baseline correction, or artifact rejection to force an
expected result.

It tests:
1. event-label consistency and class counts,
2. epoch retention imbalance,
3. signal/timing/channel consistency,
4. baseline behavior,
5. Pz versus other channels,
6. Target − NonTarget difference waves,
7. peak-window boundary effects,
8. trial-level variability,
9. grand-average and leave-one-subject-out influence.

If the clean epochs do not contain enough information to prove a cause, the notebook says
so explicitly.


## 1. Imports

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from scipy.stats import mannwhitneyu

warnings.filterwarnings("ignore", category=RuntimeWarning)

print("MNE:", mne.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)


## 2. Project Paths and Configuration

The notebook follows the same project structure as the existing Step 4 notebook and accepts
both `subject_1_clean-epo.fif` and `subject_01_clean-epo.fif`.


In [ ]:
from pathlib import Path

_current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (candidate for candidate in (_current, *_current.parents) if (candidate / "Pipfile").exists()),
    None,
)
if PROJECT_ROOT is None:
    try:
        _notebook_dir = Path(__vsc_ipynb_file__).resolve().parent
        PROJECT_ROOT = next(
            (candidate for candidate in (_notebook_dir, *_notebook_dir.parents) if (candidate / "Pipfile").exists()),
            None,
        )
    except NameError:
        pass
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root (expected Pipfile).")

DATASET_NAME = "BNCI2014-008"
PREPROCESSED_DIR = PROJECT_ROOT / "data" / "preprocessed"
RESULTS_DIR = PROJECT_ROOT / "results" / "ERP" / DATASET_NAME / "diagnostics"
FIGURES_DIR = PROJECT_ROOT / "figures" / "ERP" / DATASET_NAME / "diagnostics"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SUBJECT_IDS = list(range(1, 9))
P300_WINDOW = (0.300, 0.500)
DIAGNOSTIC_PEAK_WINDOW = (0.250, 0.600)

EXPECTED_CHANNELS = ["Fz", "Cz", "Pz", "Oz", "P3", "P4", "PO7", "PO8"]

print("Project root:", PROJECT_ROOT)
print("Preprocessed:", PREPROCESSED_DIR)


## 3. Diagnostic Helper Functions

In [ ]:
def find_epochs_file(subject_num):
    for name in [
        f"subject_{subject_num}_clean-epo.fif",
        f"subject_{subject_num:02d}_clean-epo.fif",
    ]:
        path = PREPROCESSED_DIR / name
        if path.exists():
            return path
    return None


def select_channel(ch_names):
    if "Pz" in ch_names:
        return "Pz"
    for ch in ["POz", "P3", "P4", "PO7", "PO8", "Oz"]:
        if ch in ch_names:
            return ch
    raise ValueError("No Pz or suitable parietal fallback channel found.")


def evoked_mean_uv(evoked, channel, window):
    cropped = evoked.copy().crop(tmin=window[0], tmax=window[1])
    idx = cropped.ch_names.index(channel)
    return float(np.mean(cropped.data[idx] * 1e6))


def evoked_peak_uv(evoked, channel, window):
    cropped = evoked.copy().crop(tmin=window[0], tmax=window[1])
    idx = cropped.ch_names.index(channel)
    data_uv = cropped.data[idx] * 1e6
    i = int(np.argmax(data_uv))
    return float(cropped.times[i] * 1000), float(data_uv[i])


def epoch_means_uv(epochs, channel, window):
    data = epochs.get_data(picks=[channel], tmin=window[0], tmax=window[1])
    return np.mean(data[:, 0, :] * 1e6, axis=1)


def baseline_means_uv(epochs, channel):
    data = epochs.get_data(picks=[channel], tmin=-0.2, tmax=0.0)
    return np.mean(data[:, 0, :] * 1e6, axis=1)


def cohens_d(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    if len(x) < 2 or len(y) < 2:
        return np.nan
    pooled = np.sqrt(
        ((len(x)-1)*np.var(x, ddof=1) + (len(y)-1)*np.var(y, ddof=1))
        / (len(x)+len(y)-2)
    )
    return np.nan if pooled == 0 else (np.mean(x)-np.mean(y))/pooled


def peak_at_boundary(evoked, channel, window, tolerance=1):
    cropped = evoked.copy().crop(tmin=window[0], tmax=window[1])
    idx = cropped.ch_names.index(channel)
    values = cropped.data[idx]
    peak_i = int(np.argmax(values))
    return (
        peak_i <= tolerance or peak_i >= len(values)-1-tolerance,
        float(cropped.times[peak_i] * 1000)
    )


## 4. Load All Clean Subjects

These are the outputs of the preprocessing pipeline. No additional preprocessing is done
here.


In [ ]:
subjects = {}

for sid in SUBJECT_IDS:
    path = find_epochs_file(sid)
    if path is None:
        print(f"[WARNING] Subject {sid:02d}: clean epochs file not found.")
        continue
    subjects[sid] = mne.read_epochs(path, preload=True, verbose=False)
    ep = subjects[sid]
    print(
        f"Subject {sid:02d}: {len(ep)} epochs | "
        f"{len(ep.ch_names)} channels | {ep.info['sfreq']:.1f} Hz | "
        f"{ep.tmin:.3f} to {ep.tmax:.3f} s"
    )

print(f"\nLoaded {len(subjects)}/{len(SUBJECT_IDS)} subjects.")


## 5. Event Labels and Class Counts

MOABB's BNCI2014-008 metadata maps `Target` to event code 2 and `NonTarget` to event code 1. The project preprocessing pipeline uses that same mapping. This section checks that the loaded epochs preserve those labels and codes; the label swap hypothesis is not supported if the check passes.


In [ ]:
MOABB_EVENT_ID = {"Target": 2, "NonTarget": 1}
label_rows = []

for sid, ep in subjects.items():
    ids, counts = np.unique(ep.events[:, 2], return_counts=True)
    numeric_counts = dict(zip(ids.astype(int), counts.astype(int)))
    mapping_matches_moabb = all(
        ep.event_id.get(label) == code for label, code in MOABB_EVENT_ID.items()
    )

    label_rows.append({
        "Subject": f"subject_{sid:02d}",
        "event_id mapping": str(ep.event_id),
        "Target count": len(ep["Target"]) if "Target" in ep.event_id else 0,
        "NonTarget count": len(ep["NonTarget"]) if "NonTarget" in ep.event_id else 0,
        "Total": len(ep),
        "numeric event counts": str(numeric_counts),
        "both labels present": "Target" in ep.event_id and "NonTarget" in ep.event_id,
        "mapping matches MOABB": mapping_matches_moabb,
    })

label_df = pd.DataFrame(label_rows)
display(label_df)
if not label_df["mapping matches MOABB"].all():
    raise ValueError("A clean-epoch event mapping differs from MOABB BNCI2014-008 codes.")


## 6. Epoch Retention / Imbalance

In [ ]:
# BNCI2014-008 contains 700 Target and 3,500 NonTarget events per subject.
# Comparing to this known input count makes excess retention loss visible; a
# cohort mode is misleading here because every subject has a different retained count.
INPUT_CLASS_COUNTS = {"Target": 700, "NonTarget": 3500}
retention_df = label_df[["Subject", "Target count", "NonTarget count", "Total"]].copy()
retention_df["Target rejected"] = INPUT_CLASS_COUNTS["Target"] - retention_df["Target count"]
retention_df["NonTarget rejected"] = INPUT_CLASS_COUNTS["NonTarget"] - retention_df["NonTarget count"]
retention_df["Target rejection (%)"] = 100 * retention_df["Target rejected"] / INPUT_CLASS_COUNTS["Target"]
retention_df["NonTarget rejection (%)"] = 100 * retention_df["NonTarget rejected"] / INPUT_CLASS_COUNTS["NonTarget"]
retention_df["Total rejection (%)"] = 100 * (
    retention_df["Target rejected"] + retention_df["NonTarget rejected"]
) / sum(INPUT_CLASS_COUNTS.values())

display(retention_df.round(2))


## 7. Signal, Timing, Channels, Units, and Reference Metadata

In [ ]:
consistency_rows = []

for sid, ep in subjects.items():
    data_uv = ep.get_data() * 1e6
    consistency_rows.append({
        "Subject": f"subject_{sid:02d}",
        "Channels": ", ".join(ep.ch_names),
        "Expected channels all present": all(ch in ep.ch_names for ch in EXPECTED_CHANNELS),
        "Sampling rate (Hz)": ep.info["sfreq"],
        "Epoch start (s)": ep.tmin,
        "Epoch end (s)": ep.tmax,
        "Data min (uV)": float(np.min(data_uv)),
        "Data max (uV)": float(np.max(data_uv)),
        "Bad channels": ", ".join(ep.info.get("bads", [])) or "None",
        "custom_ref_applied": ep.info.get("custom_ref_applied"),
    })

consistency_df = pd.DataFrame(consistency_rows)
display(consistency_df)


## 8. Baseline Check at Pz

In [ ]:
baseline_rows = []

for sid, ep in subjects.items():
    if "Pz" not in ep.ch_names:
        continue

    target_base = baseline_means_uv(ep["Target"], "Pz")
    nontarget_base = baseline_means_uv(ep["NonTarget"], "Pz")

    baseline_rows.append({
        "Subject": f"subject_{sid:02d}",
        "Target baseline mean (uV)": np.mean(target_base),
        "Target baseline SD (uV)": np.std(target_base, ddof=1),
        "NonTarget baseline mean (uV)": np.mean(nontarget_base),
        "NonTarget baseline SD (uV)": np.std(nontarget_base, ddof=1),
        "Max abs Target epoch baseline mean (uV)": np.max(np.abs(target_base)),
        "Max abs NonTarget epoch baseline mean (uV)": np.max(np.abs(nontarget_base)),
    })

baseline_df = pd.DataFrame(baseline_rows)
display(baseline_df)


## 9. Reproduce the Per-Subject ERP Sanity Check

The diagnostic rule is the same one used in the current Step 4 notebook:
Target must have both a larger 300–500 ms mean and a larger positive peak than NonTarget.

This is an implementation check, **not a formal requirement that every subject must pass**.


In [ ]:
erp_rows = []
evoked = {}

for sid, ep in subjects.items():
    if "Target" not in ep.event_id or "NonTarget" not in ep.event_id:
        continue

    channel = select_channel(ep.ch_names)
    target_evoked = ep["Target"].average()
    nontarget_evoked = ep["NonTarget"].average()

    evoked[sid] = (target_evoked, nontarget_evoked, channel)

    t_mean = evoked_mean_uv(target_evoked, channel, P300_WINDOW)
    n_mean = evoked_mean_uv(nontarget_evoked, channel, P300_WINDOW)

    t_lat, t_peak = evoked_peak_uv(target_evoked, channel, P300_WINDOW)
    n_lat, n_peak = evoked_peak_uv(nontarget_evoked, channel, P300_WINDOW)

    mean_pass = t_mean > n_mean
    peak_pass = t_peak > n_peak

    erp_rows.append({
        "Subject": f"subject_{sid:02d}",
        "Target epochs": len(ep["Target"]),
        "NonTarget epochs": len(ep["NonTarget"]),
        "Channel": channel,
        "Target mean 300-500ms (uV)": t_mean,
        "NonTarget mean 300-500ms (uV)": n_mean,
        "Target-NonTarget mean (uV)": t_mean - n_mean,
        "Target peak latency (ms)": t_lat,
        "Target peak amplitude (uV)": t_peak,
        "NonTarget peak latency (ms)": n_lat,
        "NonTarget peak amplitude (uV)": n_peak,
        "Mean check": mean_pass,
        "Peak check": peak_pass,
        "Sanity result": "EXPECTED_PATTERN" if mean_pass and peak_pass else "UNEXPECTED_PATTERN",
    })

erp_df = pd.DataFrame(erp_rows)
display(erp_df)


In [ ]:
unexpected_subjects = erp_df.loc[
    erp_df["Sanity result"] == "UNEXPECTED_PATTERN", "Subject"
].tolist()

expected_subjects = erp_df.loc[
    erp_df["Sanity result"] == "EXPECTED_PATTERN", "Subject"
].tolist()

print("Expected:", expected_subjects)
print("Unexpected:", unexpected_subjects)


## 10. Pz vs Other Channels — Is the Result Spatially Specific?

In [ ]:
channel_rows = []

for sid, ep in subjects.items():
    target_evoked = ep["Target"].average()
    nontarget_evoked = ep["NonTarget"].average()

    for ch in ep.ch_names:
        t = evoked_mean_uv(target_evoked, ch, P300_WINDOW)
        n = evoked_mean_uv(nontarget_evoked, ch, P300_WINDOW)
        channel_rows.append({
            "Subject": f"subject_{sid:02d}",
            "Channel": ch,
            "Target mean (uV)": t,
            "NonTarget mean (uV)": n,
            "Target-NonTarget (uV)": t - n,
        })

channel_df = pd.DataFrame(channel_rows)

channel_summary = []
for subject in channel_df["Subject"].unique():
    sub = channel_df[channel_df["Subject"] == subject]
    pz = sub[sub["Channel"] == "Pz"]["Target-NonTarget (uV)"]
    pz_diff = float(pz.iloc[0]) if len(pz) else np.nan

    channel_summary.append({
        "Subject": subject,
        "Positive channels": int((sub["Target-NonTarget (uV)"] > 0).sum()),
        "Total channels": len(sub),
        "Positive-channel fraction": float((sub["Target-NonTarget (uV)"] > 0).mean()),
        "Pz difference (uV)": pz_diff,
        "Median channel difference (uV)": float(sub["Target-NonTarget (uV)"].median()),
    })

channel_summary_df = pd.DataFrame(channel_summary)

print("All-channel results for the unexpected subjects:")
display(channel_df[channel_df["Subject"].isin(unexpected_subjects)])

print("Compact spatial summary:")
display(channel_summary_df[channel_summary_df["Subject"].isin(unexpected_subjects)])


## 11. Difference Waves for the Unexpected Subjects

A negative Target − NonTarget difference throughout the 300–500 ms region indicates that
the unexpected result is genuinely present in the selected waveform. The shape also shows
whether it is a broad reversal or a small local feature.


In [ ]:
for subject_label in unexpected_subjects:
    sid = int(subject_label.split("_")[1])
    target_evoked, nontarget_evoked, channel = evoked[sid]
    idx = target_evoked.ch_names.index(channel)

    times_ms = target_evoked.times * 1000
    diff_uv = (target_evoked.data[idx] - nontarget_evoked.data[idx]) * 1e6

    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(times_ms, diff_uv, linewidth=1.8, label="Target − NonTarget")
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.axvline(0, linestyle="--", linewidth=1)
    ax.axvspan(300, 500, alpha=0.2, label="300–500 ms")
    ax.set_title(f"{subject_label} — Difference wave at {channel}")
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Amplitude difference (uV)")
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()

    path = FIGURES_DIR / f"{DATASET_NAME}_{subject_label}_difference_wave.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved:", path)


## 12. Peak Detection Diagnostic

The existing Step 4 notebook searches for the positive peak **inside 300–500 ms**.
If a peak lands exactly at 500 ms, it may be a window-edge effect.

This cell uses a wider **250–600 ms diagnostic window**. It does not replace the Step 4
measurement; it tests whether the reported peak is being truncated by the analysis window.


In [ ]:
peak_rows = []

for sid, (target_evoked, nontarget_evoked, channel) in evoked.items():
    for condition, e in [("Target", target_evoked), ("NonTarget", nontarget_evoked)]:
        lat_300, amp_300 = evoked_peak_uv(e, channel, P300_WINDOW)
        edge_300, _ = peak_at_boundary(e, channel, P300_WINDOW)

        lat_wide, amp_wide = evoked_peak_uv(e, channel, DIAGNOSTIC_PEAK_WINDOW)
        edge_wide, _ = peak_at_boundary(e, channel, DIAGNOSTIC_PEAK_WINDOW)

        channel_idx = e.ch_names.index(channel)
        late_mask = (e.times > 0.500) & (e.times <= 0.800)
        full_mask = (e.times >= 0.300) & (e.times <= 0.800)
        late_mean_uv = float(np.mean(e.data[channel_idx, late_mask]) * 1e6)
        full_mean_uv = float(np.mean(e.data[channel_idx, full_mask]) * 1e6)

        peak_rows.append({
            "Subject": f"subject_{sid:02d}",
            "Condition": condition,
            "Mean amplitude 500-800 ms (uV)": late_mean_uv,
            "Mean amplitude 300-800 ms (uV)": full_mean_uv,
            "300-500 peak latency (ms)": lat_300,
            "300-500 peak amp (uV)": amp_300,
            "300-500 peak at boundary": edge_300,
            "250-600 peak latency (ms)": lat_wide,
            "250-600 peak amp (uV)": amp_wide,
            "250-600 peak at boundary": edge_wide,
        })

peak_df = pd.DataFrame(peak_rows)
display(peak_df[peak_df["Subject"].isin(unexpected_subjects)])


## 13. Quantify Response After 500 ms

The peak search suggests some unexpected Target peaks occur at or after 500 ms. This comparison checks whether the Target–NonTarget mean difference becomes positive after 500 ms, and whether widening the measurement window changes the result. It is a diagnostic comparison, not a replacement for the specified 300–500 ms result.


In [ ]:
late_rows = []
for sid, (target_evoked, nontarget_evoked, channel) in evoked.items():
    ch = target_evoked.ch_names.index(channel)
    late_mask = (target_evoked.times > 0.500) & (target_evoked.times <= 0.800)
    full_mask = (target_evoked.times >= 0.300) & (target_evoked.times <= 0.800)
    target_late = float(np.mean(target_evoked.data[ch, late_mask]) * 1e6)
    nontarget_late = float(np.mean(nontarget_evoked.data[ch, late_mask]) * 1e6)
    target_full = float(np.mean(target_evoked.data[ch, full_mask]) * 1e6)
    nontarget_full = float(np.mean(nontarget_evoked.data[ch, full_mask]) * 1e6)
    late_rows.append({
        "Subject": f"subject_{sid:02d}",
        "Target-NonTarget mean 300-500 ms (uV)": evoked_mean_uv(target_evoked, channel, P300_WINDOW) - evoked_mean_uv(nontarget_evoked, channel, P300_WINDOW),
        "Target-NonTarget mean 500-800 ms (uV)": target_late - nontarget_late,
        "Target-NonTarget mean 300-800 ms (uV)": target_full - nontarget_full,
        "Target mean 500-800 ms (uV)": target_late,
        "NonTarget mean 500-800 ms (uV)": nontarget_late,
    })
late_window_df = pd.DataFrame(late_rows)
display(late_window_df.round(3))


## 14. Trial-Level Variability

Because an ERP is an average, a weak or reversed subject-level ERP can result from high
trial-to-trial variability.

We therefore inspect the distribution of each epoch's mean Pz amplitude in 300–500 ms.
Cohen's d and a Mann–Whitney p-value are descriptive diagnostics, not classifier metrics.


In [ ]:
trial_rows = []

for sid, ep in subjects.items():
    target_values = epoch_means_uv(ep["Target"], "Pz", P300_WINDOW)
    nontarget_values = epoch_means_uv(ep["NonTarget"], "Pz", P300_WINDOW)

    try:
        p_value = float(
            mannwhitneyu(
                target_values,
                nontarget_values,
                alternative="two-sided"
            ).pvalue
        )
    except Exception:
        p_value = np.nan

    trial_rows.append({
        "Subject": f"subject_{sid:02d}",
        "Target n": len(target_values),
        "Target trial mean (uV)": np.mean(target_values),
        "Target trial SD (uV)": np.std(target_values, ddof=1),
        "NonTarget n": len(nontarget_values),
        "NonTarget trial mean (uV)": np.mean(nontarget_values),
        "NonTarget trial SD (uV)": np.std(nontarget_values, ddof=1),
        "Cohen d": cohens_d(target_values, nontarget_values),
        "Mann-Whitney p": p_value,
    })

trial_df = pd.DataFrame(trial_rows)
display(trial_df[trial_df["Subject"].isin(unexpected_subjects)])


In [ ]:
for subject_label in unexpected_subjects:
    sid = int(subject_label.split("_")[1])
    ep = subjects[sid]

    t = epoch_means_uv(ep["Target"], "Pz", P300_WINDOW)
    n = epoch_means_uv(ep["NonTarget"], "Pz", P300_WINDOW)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.hist(t, bins=40, alpha=0.55, label="Target")
    ax.hist(n, bins=40, alpha=0.55, label="NonTarget")
    ax.axvline(np.mean(t), linestyle="--", linewidth=1.5, label="Target mean")
    ax.axvline(np.mean(n), linestyle="--", linewidth=1.5, label="NonTarget mean")
    ax.set_title(f"{subject_label} — Pz trial means in 300–500 ms")
    ax.set_xlabel("Mean amplitude per epoch (uV)")
    ax.set_ylabel("Epoch count")
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()

    path = FIGURES_DIR / f"{DATASET_NAME}_{subject_label}_trial_distribution.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved:", path)


## 15. Grand Average and Leave-One-Subject-Out Influence

This checks whether any individual subject strongly changes the group-level result.

A large leave-one-out change identifies an **influential subject**, not necessarily a faulty
subject.


In [ ]:
target_evokeds = [v[0] for v in evoked.values()]
nontarget_evokeds = [v[1] for v in evoked.values()]

grand_target = mne.grand_average(target_evokeds, interpolate_bads=False, drop_bads=False)
grand_nontarget = mne.grand_average(nontarget_evokeds, interpolate_bads=False, drop_bads=False)

grand_channel = "Pz" if "Pz" in grand_target.ch_names else select_channel(grand_target.ch_names)

grand_target_mean = evoked_mean_uv(grand_target, grand_channel, P300_WINDOW)
grand_nontarget_mean = evoked_mean_uv(grand_nontarget, grand_channel, P300_WINDOW)
grand_diff = grand_target_mean - grand_nontarget_mean

print("Grand channel:", grand_channel)
print(f"Grand Target mean: {grand_target_mean:.4f} uV")
print(f"Grand NonTarget mean: {grand_nontarget_mean:.4f} uV")
print(f"Grand Target - NonTarget: {grand_diff:.4f} uV")


In [ ]:
loo_rows = []

for excluded in sorted(evoked):
    remaining = [sid for sid in sorted(evoked) if sid != excluded]

    t = mne.grand_average(
        [evoked[sid][0] for sid in remaining],
        interpolate_bads=False,
        drop_bads=False
    )
    n = mne.grand_average(
        [evoked[sid][1] for sid in remaining],
        interpolate_bads=False,
        drop_bads=False
    )

    t_mean = evoked_mean_uv(t, grand_channel, P300_WINDOW)
    n_mean = evoked_mean_uv(n, grand_channel, P300_WINDOW)
    diff = t_mean - n_mean

    loo_rows.append({
        "Excluded subject": f"subject_{excluded:02d}",
        "LOO difference (uV)": diff,
        "Change from full grand difference (uV)": diff - grand_diff,
    })

loo_df = pd.DataFrame(loo_rows)
display(loo_df)


## 16. Visual Comparison of Unexpected Subjects

In [ ]:
fig, axes = plt.subplots(
    len(unexpected_subjects), 1,
    figsize=(10, 3.4 * max(1, len(unexpected_subjects))),
    sharex=True
)

if len(unexpected_subjects) == 1:
    axes = [axes]

for ax, subject_label in zip(axes, unexpected_subjects):
    sid = int(subject_label.split("_")[1])
    target_evoked, nontarget_evoked, channel = evoked[sid]
    idx = target_evoked.ch_names.index(channel)
    times_ms = target_evoked.times * 1000

    ax.plot(times_ms, target_evoked.data[idx] * 1e6, label="Target", linewidth=1.6)
    ax.plot(times_ms, nontarget_evoked.data[idx] * 1e6, label="NonTarget", linewidth=1.6)
    ax.axvline(0, linestyle="--", linewidth=1)
    ax.axvspan(300, 500, alpha=0.2)

    ax.set_title(f"{subject_label} — {channel}")
    ax.set_ylabel("uV")
    ax.legend()
    ax.grid(alpha=0.25)

axes[-1].set_xlabel("Time (ms)")
fig.suptitle("Unexpected Subjects — Target vs NonTarget ERP Analysis", y=1.01)
fig.tight_layout()

path = FIGURES_DIR / f"{DATASET_NAME}_unexpected_subjects_erp_comparison.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", path)


## 17. Evidence-Based Diagnostic Summary

The analysis tests whether the observed Pz result is associated with event-label errors, unusually high epoch rejection, spatial/topographic differences, or delayed peaks. These findings identify what is present in the data; they do not prove a biological cause.


In [ ]:
diagnostic_rows = []

for subject_label in unexpected_subjects:
    er = erp_df[erp_df["Subject"] == subject_label].iloc[0]
    cs = channel_summary_df[channel_summary_df["Subject"] == subject_label].iloc[0]
    tr = trial_df[trial_df["Subject"] == subject_label].iloc[0]
    pk = peak_df[peak_df["Subject"] == subject_label]
    retention = retention_df[retention_df["Subject"] == subject_label].iloc[0]

    positive_fraction = cs["Positive-channel fraction"]
    if er["Target-NonTarget mean (uV)"] < 0 and positive_fraction > 0.5:
        spatial = "Pz negative, most other channels positive"
    elif er["Target-NonTarget mean (uV)"] < 0:
        spatial = "Negative difference across most channels"
    else:
        spatial = "No negative Pz difference"

    target_peak_late = bool(
        ((pk["Condition"] == "Target")
        & (pk["250-600 peak latency (ms)"] > 500)).any()
    )
    late = late_window_df[late_window_df["Subject"] == subject_label].iloc[0]

    diagnostic_rows.append({
        "Subject": subject_label,
        "Target rejected": retention["Target rejected"],
        "NonTarget rejected": retention["NonTarget rejected"],
        "Total rejection (%)": retention["Total rejection (%)"],
        "Pz difference 300-500 ms (uV)": er["Target-NonTarget mean (uV)"],
        "Positive-channel fraction": positive_fraction,
        "Spatial finding": spatial,
        "Target peak after 500 ms": target_peak_late,
        "Target-NonTarget difference 500-800 ms (uV)": late["Target-NonTarget mean 500-800 ms (uV)"],
        "Target-NonTarget difference 300-800 ms (uV)": late["Target-NonTarget mean 300-800 ms (uV)"],
        "Peak at 300-500 boundary": bool(pk["300-500 peak at boundary"].any()),
        "Cohen d": tr["Cohen d"],
        "Mann-Whitney p": tr["Mann-Whitney p"],
    })

diagnostic_df = pd.DataFrame(diagnostic_rows)
display(diagnostic_df.round(3))


## 18. Interpretation — What the Diagnostic Evidence Shows


In [ ]:
print("=" * 80)
print("FINAL DIAGNOSTIC REPORT")
print("=" * 80)
print(f"Subjects analyzed: {len(erp_df)} | expected pattern: {len(expected_subjects)} | unexpected pattern: {len(unexpected_subjects)}")

print("\nA. EVENT LABELS")
print("MOABB Target=2 / NonTarget=1 mapping matches all clean epochs:", bool(label_df["mapping matches MOABB"].all()))

print("\nB. RETENTION")
worst = retention_df.loc[retention_df["Total rejection (%)"].idxmax()]
print(f"Highest rejection: {worst['Subject']} — {worst['Total rejection (%)']:.2f}% "
      f"({int(worst['Target rejected'])} Target + {int(worst['NonTarget rejected'])} NonTarget).")
print("This is a possible data-quality contributor for that subject, not a demonstrated cause of the cohort Pz result.")

print("\nC. GRAND-AVERAGE CHANNEL DIFFERENCES (Target − NonTarget, 300–500 ms)")
group_channel_rows = []
for channel in grand_target.ch_names:
    t_mean = evoked_mean_uv(grand_target, channel, P300_WINDOW)
    n_mean = evoked_mean_uv(grand_nontarget, channel, P300_WINDOW)
    group_channel_rows.append({
        "Channel": channel,
        "Target mean (uV)": t_mean,
        "NonTarget mean (uV)": n_mean,
        "Target-NonTarget mean (uV)": t_mean - n_mean,
    })
group_channel_df = pd.DataFrame(group_channel_rows)
display(group_channel_df.round(3))
for _, row in group_channel_df.iterrows():
    print(f"{row['Channel']}: {row['Target-NonTarget mean (uV)']:+.3f} uV")

print("\nD. INDIVIDUAL SPATIAL AND TEMPORAL FINDINGS")
for _, row in diagnostic_df.iterrows():
    print(f"{row['Subject']}: {row['Spatial finding']}; "
          f"Pz 300-500 ms={row['Pz difference 300-500 ms (uV)']:.3f} uV; "
          f"Pz 500-800 ms={row['Target-NonTarget difference 500-800 ms (uV)']:.3f} uV; "
          f"Target peak after 500 ms={row['Target peak after 500 ms']}.")

print("\nE. WHAT THE EVIDENCE SUPPORTS")
print("The Target/NonTarget mapping is correct; the results do not support a class-code swap.")
print("The negative cohort Pz difference is spatially specific at group level: the same grand average is positive at Fz/Cz but negative at Pz and several posterior electrodes.")
print("Subjects 02, 05, and 08 show negative differences across most channels; Subject 07 is spatially mixed and also has 18.57% epoch rejection.")
print("Target peaks in all four unexpected subjects are after 500 ms in the wider search. The 500-800 ms difference becomes positive for Subjects 05 and 07, but remains negative across 300-800 ms for Subjects 02 and 08.")
print("So channel topography and subject-level timing/amplitude variation explain the observed pattern descriptively; delayed responses contribute for some subjects. The data do not establish one underlying biological cause or a single preprocessing bug.")


## 19. Save Diagnostic Tables

In [ ]:
tables = {
    f"{DATASET_NAME}_01_label_and_counts.csv": label_df,
    f"{DATASET_NAME}_02_retention_analysis.csv": retention_df,
    f"{DATASET_NAME}_03_signal_timing_channel_consistency.csv": consistency_df,
    f"{DATASET_NAME}_04_baseline_analysis.csv": baseline_df,
    f"{DATASET_NAME}_05_subject_erp_measurements.csv": erp_df,
    f"{DATASET_NAME}_06_channel_level_differences.csv": channel_df,
    f"{DATASET_NAME}_07_channel_summary.csv": channel_summary_df,
    f"{DATASET_NAME}_08_peak_window_diagnostic.csv": peak_df,
    f"{DATASET_NAME}_08b_post_500ms_window_comparison.csv": late_window_df,
    f"{DATASET_NAME}_09_trial_level_variability.csv": trial_df,
    f"{DATASET_NAME}_10_leave_one_subject_out.csv": loo_df,
    f"{DATASET_NAME}_11_final_diagnostic_summary.csv": diagnostic_df,
    f"{DATASET_NAME}_12_group_channel_comparison.csv": group_channel_df,
}

for filename, df in tables.items():
    df.to_csv(RESULTS_DIR / filename, index=False)

print("Saved diagnostic tables:")
for filename in tables:
    print(" -", RESULTS_DIR / filename)
print("\nSaved figures:")
for path in sorted(FIGURES_DIR.glob("*.png")):
    print(" -", path)


# What to Report in the Project

Report the refreshed Pz result and the channel-dependent findings. The cohort Target–NonTarget mean difference is negative at Pz (−0.059 µV) but positive at Fz (+0.987 µV) and Cz (+0.629 µV). This is a spatially varying group result, not a uniform reversal. Several unexpected subjects also have Target peaks after 500 ms; the post-500 ms response changes the interpretation for Subjects 05 and 07, but does not remove the negative 300–800 ms difference for Subjects 02 and 08.

The event-code mapping matches MOABB. Subject 07 has unusually high rejection (18.57%), which warrants separate review but cannot account for the pattern across all unexpected subjects. The available evidence supports subject-level ERP timing and topographic variability; it does not identify a single confirmed cause.
